# IDK time benchmarking

Run each combination of the `_GRID` parameters and time **fit**, **embed**, and **similarity**. Data generation and model construction are outside the timers. Results stay in memory and appear in the table below.

Use the project's `nb` environment. Large grid settings can take a long time.


In [ ]:
from itertools import product
from time import perf_counter

import numpy as np
from IPython.display import Markdown, display

from pyidk import IsolationDistributionalKernel, IsolationKernel, SequenceBatch

SEED = 42
N_PARTITIONS_GRID = [25, 100, 250]
PSI_FRACTIONS_GRID = [0.0005, 0.005]
SAMPLE_COUNTS_GRID = [2**10, 2**16, 2**18, 2**20]  # 1K, 65K, 262K, 1M
FEATURE_COUNTS_GRID = [2, 4, 16]
N_DISTRIBUTIONS = 16  # Independent groups/episodes
REPEATS = 1
CHUNK_SIZE = 4096


def psi_for(n_samples: int, psi_fraction: float) -> int:
    return min(n_samples, max(2, int(np.ceil(psi_fraction * n_samples))))

## Terminology

- **N** is the total number of observations
- **d** is input feature dimension
- **t** is the number of partitions
- **psi** is the number of centers sampled per partition

In [ ]:
configurations = list(
    product(
        SAMPLE_COUNTS_GRID,
        FEATURE_COUNTS_GRID,
        N_PARTITIONS_GRID,
        PSI_FRACTIONS_GRID,
    )
)
print(f"{len(configurations)} configurations, {REPEATS} repetition(s) each.")

## Execution Time Contributors

Here are the steps that contribute to time complexity:

- **Fit** 
- **Embed** 
- **Similarity** 

Some research needs to be conducted to determine precise complexity.

## Benchmarking Functions


In [ ]:
def make_dataset(n_samples: int, n_features: int) -> SequenceBatch:
    if n_samples < N_DISTRIBUTIONS:
        raise ValueError("Every distribution must contain at least one observation.")
    rng = np.random.default_rng(np.random.SeedSequence([SEED, n_samples, n_features]))
    values = rng.standard_normal((n_samples, n_features))
    # Balanced groups cover all rows even when division has a remainder.
    offsets = np.arange(N_DISTRIBUTIONS + 1, dtype=np.int64) * n_samples // N_DISTRIBUTIONS
    return SequenceBatch(values, offsets)


def benchmark_once(data: SequenceBatch, psi: int, n_partitions: int) -> dict[str, float]:
    model = IsolationDistributionalKernel(
        IsolationKernel(
            n_partitions=n_partitions,
            samples_per_partition=psi,
            random_state=SEED,
            chunk_size=CHUNK_SIZE,
        )
    )
    start = perf_counter()
    model.fit(data)
    fitted = perf_counter()
    embeddings = model.transform(data)
    embedded = perf_counter()
    model.similarity(embeddings)
    finished = perf_counter()

    return {
        "fit_s": fitted - start,
        "embed_s": embedded - fitted,
        "similarity_s": finished - embedded,
        "total_s": finished - start,
    }

## Run the grid

Each repetition uses a fresh model. The loop prints elapsed time after each configuration.


In [ ]:
records = []
for _ in range(REPEATS):
    for n, d, t, fraction in configurations:
        data = make_dataset(n, d)
        psi = psi_for(n, fraction)
        result = benchmark_once(data, psi, t)
        records.append({"n": n, "d": d, "t": t, "fraction": fraction, "psi": psi, **result})
        print(f"N={n:,}, d={d}, t={t}, psi={psi:,}: {result['total_s']:.3f} s", flush=True)

## Results table

Times are medians across repetitions, in seconds. With one repetition, each entry is the measured time.


In [ ]:
rows = []
for n, d, t, fraction in configurations:
    runs = [r for r in records if (r["n"], r["d"], r["t"], r["fraction"]) == (n, d, t, fraction)]
    if not runs:
        continue
    times = [
        np.median([r[key] for r in runs]) for key in ["fit_s", "embed_s", "similarity_s", "total_s"]
    ]
    rows.append(
        f"| {n:,} | {d} | {t} | {fraction:g} | {psi_for(n, fraction):,} | "
        + " | ".join(f"{value:.4f}" for value in times)
        + " |"
    )
display(
    Markdown(
        "| N | d | t | Fraction | psi | Fit s | Embed s | Similarity s | Total s |\n"
        "|--:|--:|--:|--:|--:|--:|--:|--:|--:|\n" + "\n".join(rows)
    )
)